# EfficientNet-B2 Emotion Recognition — FER2013 Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

Steps:
1. Check GPU
2. Upload or load `fer2013.zip` from Google Drive
3. Extract dataset
4. Train EfficientNet-B2 (Phase 1: head only → Phase 2: fine-tune top layers)
5. Save and download `emotion_efficientnet.keras`

## 1 — Check GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs available:', len(gpus))
for g in gpus:
    print(' ', tf.config.experimental.get_device_details(g))

if not gpus:
    print('\n⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU and re-run.')
else:
    print('\n✅ GPU ready.')

## 2 — Load Dataset

**Run ONE of the two cells below** depending on where your `fer2013.zip` is.

In [ ]:
# ── Option A: Upload fer2013.zip directly ──────────────────────────────────
# Run this cell, click 'Choose Files', and select fer2013.zip from your machine.

from google.colab import files
uploaded = files.upload()  # select fer2013.zip

import shutil
shutil.move(list(uploaded.keys())[0], '/content/fer2013.zip')
print('Uploaded to /content/fer2013.zip')

In [ ]:
# ── Option B: Load from Google Drive ──────────────────────────────────────
# Put fer2013.zip anywhere in your Drive, update ZIP_PATH below, then run.

ZIP_PATH = '/content/drive/MyDrive/fer2013.zip'  # ← update if needed

from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(ZIP_PATH, '/content/fer2013.zip')
print('Copied to /content/fer2013.zip')

## 3 — Extract & Verify Dataset

In [ ]:
import zipfile, os

print('Extracting...')
with zipfile.ZipFile('/content/fer2013.zip', 'r') as z:
    z.extractall('/content/')
print('Done.')

# Verify structure
base = '/content/datasets/fer2013'
print()
for split in ['train', 'val', 'test']:
    total = 0
    for cls in sorted(os.listdir(f'{base}/{split}')):
        n = len(os.listdir(f'{base}/{split}/{cls}'))
        total += n
    print(f'{split}: {total} images')

## 4 — Define Model & Data Pipeline

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ── CONFIG ────────────────────────────────────────────────────────────────
TRAIN_DIR  = '/content/datasets/fer2013/train'
VAL_DIR    = '/content/datasets/fer2013/val'
IMG_SIZE   = 48
BATCH_SIZE = 64
SAVE_PATH  = '/content/emotion_efficientnet.keras'

# ── DATA GENERATORS ──────────────────────────────────────────────────────
# preprocess_input expects [0,255] — EfficientNet normalises internally
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
)
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True,
)
val_data = val_datagen.flow_from_directory(
    VAL_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)

print('Class index mapping:', train_data.class_indices)
print(f'Train batches: {len(train_data)}  |  Val batches: {len(val_data)}')

In [ ]:
# ── MODEL DEFINITION ─────────────────────────────────────────────────────
base = EfficientNetB2(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base.trainable = False  # frozen for phase 1

x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
output = layers.Dense(7, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)

trainable = sum(1 for l in model.layers if l.trainable)
total     = len(model.layers)
print(f'Layers: {total} total, {trainable} trainable (head only)')

## 5 — Train

**Phase 1** trains only the new classification head (base frozen, faster).  
**Phase 2** unfreezes the top 30 base layers for fine-tuning (slower, better accuracy).

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

# ── PHASE 1: train head only ──────────────────────────────────────────────
print('\n=== Phase 1: training classification head (base frozen) ===')
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
history_p1 = model.fit(
    train_data, validation_data=val_data,
    epochs=10, callbacks=callbacks,
)

# ── PHASE 2: fine-tune top 30 base layers ────────────────────────────────
print('\n=== Phase 2: fine-tuning top 30 base layers ===')
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

trainable = sum(1 for l in model.layers if l.trainable)
print(f'{trainable} trainable layers now')

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
history_p2 = model.fit(
    train_data, validation_data=val_data,
    epochs=15, callbacks=callbacks,
)

## 6 — Evaluate on Test Set

In [ ]:
TEST_DIR = '/content/datasets/fer2013/test'

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_data = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb', batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False,
)

loss, acc = model.evaluate(test_data)
print(f'\nTest loss: {loss:.4f}  |  Test accuracy: {acc:.4f}')

## 7 — Save & Download

In [ ]:
model.save(SAVE_PATH)
print(f'Model saved to {SAVE_PATH}')

import os
size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f'File size: {size_mb:.1f} MB')

# Download to local machine
from google.colab import files
files.download(SAVE_PATH)
print('Download started.')

## (Optional) Save to Google Drive instead

Run this cell if you prefer to save to Drive rather than downloading directly.

In [ ]:
import shutil
drive_path = '/content/drive/MyDrive/emotion_efficientnet.keras'
shutil.copy(SAVE_PATH, drive_path)
print(f'Saved to Google Drive: {drive_path}')